<a href="https://colab.research.google.com/github/santhiago-svb/Statistical-Learning-e22047/blob/main/E22047_ASSIGNMENT_6_GPR_LR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.metrics import r2_score, mean_squared_error

# Download latest version
kagglepath = "elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

# Listing contents of dataset directory
print(f"Listing contents of: {path}")
!ls {path}

# Read the data file
df2 = pd.read_csv(path + "/ENB2012_data.csv")

# Clean column names to prevent unexpected whitespace indexing errors
df2.columns = df2.columns.str.strip()

# -------------------------------------------------------------
# Data Preprocessing & Splitting
# X1 to X8: Structural attributes | Y1, Y2: Thermal Loads
# -------------------------------------------------------------
X = df2[['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']].values
Y = df2[['Y1', 'Y2']].values

# Split data into 80% Training and 20% Testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Scale continuous structural features for numerical stability in GPR
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- FIXED LINE 43: Corrected argument names for Strategy 1 ---
base_kernel = (
    C(1.0, constant_value_bounds=(1e-2, 1e2)) * RBF(length_scale=np.ones(8), length_scale_bounds=(1e-2, 1e2)) +
    WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))
)

# =============================================================
# STRATEGY 1: Natively Multi-Output GP (Shared Hyperparameters)
# =============================================================
print("\n=== Strategy 1: Shared Hyperparameter Multi-Output GP ===")
gp_multi = GaussianProcessRegressor(kernel=base_kernel, n_restarts_optimizer=5, random_state=42)
gp_multi.fit(X_train_scaled, Y_train)

# Predict targets
Y_pred_multi = gp_multi.predict(X_test_scaled)

# Evaluate each target independently
for i, name in enumerate(['Heating Load (Y1)', 'Cooling Load (Y2)']):
    r2 = r2_score(Y_test[:, i], Y_pred_multi[:, i])
    mse = mean_squared_error(Y_test[:, i], Y_pred_multi[:, i])
    print(f"{name} -> R2 Score: {r2:.4f} | MSE: {mse:.4f}")

# Optimized kernel parameters
print("Optimized Hyperparameters (Shared):", gp_multi.kernel_)


# =============================================================
# STRATEGY 2: Single Augmented GP (Stacked via Indicator Variable)
# =============================================================
print("\n=== Strategy 2: Augmented Input Space Scalar GP ===")
n_train = X_train.shape[0]
n_test = X_test.shape[0]

# Stack features and append binary task indicator column (0 = Heating, 1 = Cooling)
X_train_aug = np.vstack([
    np.hstack([X_train_scaled, np.zeros((n_train, 1))]),
    np.hstack([X_train_scaled, np.ones((n_train, 1))])
])
y_train_aug = np.hstack([Y_train[:, 0], Y_train[:, 1]])

X_test_aug = np.vstack([
    np.hstack([X_test_scaled, np.zeros((n_test, 1))]),
    np.hstack([X_test_scaled, np.ones((n_test, 1))])
])
y_test_aug = np.hstack([Y_test[:, 0], Y_test[:, 1]])

# --- FIXED LINE 81: Corrected argument names for Strategy 2 (9-Dimensional) ---
aug_kernel = (
    C(1.0, constant_value_bounds=(1e-2, 1e2)) * RBF(length_scale=np.ones(9), length_scale_bounds=(1e-2, 1e2)) +
    WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))
)

gp_single = GaussianProcessRegressor(kernel=aug_kernel, n_restarts_optimizer=5, random_state=42)
gp_single.fit(X_train_aug, y_train_aug)

# Predict on the combined test space
y_pred_aug = gp_single.predict(X_test_aug)

# Split predictions back to evaluate performance metrics
y_pred_heating = y_pred_aug[:n_test]
y_pred_cooling = y_pred_aug[n_test:]

print(f"Heating Load (Y1) -> R2 Score: {r2_score(Y_test[:, 0], y_pred_heating):.4f} | MSE: {mean_squared_error(Y_test[:, 0], y_pred_heating):.4f}")
print(f"Cooling Load (Y2) -> R2 Score: {r2_score(Y_test[:, 1], y_pred_cooling):.4f} | MSE: {mean_squared_error(Y_test[:, 1], y_pred_cooling):.4f}")
print("Optimized Hyperparameters (Augmented Space):", gp_single.kernel_)

# Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

In [ ]:
import os
import numpy as np
import pandas as pd
import kagglehub
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 1. Download latest version of the dataset
kagglepath = "programmer3/green-building-multi-source-environment-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)
print(f"Listing contents of: {path}")
!ls {path}

# 2. Load the dataset into a DataFrame
df2 = pd.read_csv(path + "/green_building_dataset.csv")

# Placeholder interface class matching 'inspector.df=df2' structure if required by your notebook
class DataInspector:
    def _init_(self):
        self.df = None
inspector = DataInspector()
inspector.df = df2

# 3. Exploration & Data Processing
print("\n--- Dataset Summary ---")
print(df2.info())
print("\nMissing values per column:\n", df2.isnull().sum())

# Define variables based on typical multi-source green building attributes
# Target variable: 'predicted_energy_demand'
target = 'predicted_energy_demand'

# Note: Based on multi-source data parameters (indoor conditions, outdoor weather, occupancy)
# We assume standard names present in environmental IoT datasets such as:
# ['indoor_temperature', 'outdoor_temperature', 'humidity', 'occupancy_count', 'lighting_intensity']
# If the exact schema names vary slightly, select corresponding numerical environmental factors from df2.columns.
potential_features = [col for col in df2.columns if col != target and df2[col].dtype in [np.float64, np.int64]]

print(f"\nAutomatically selected numerical parameters for exploration:\n{potential_features}")

# Filter out non-contributing or ID columns if any exist
features = [f for f in potential_features if f.lower() not in ['id', 'timestamp']]

# Drop rows with missing values in our subset (if any)
df_clean = df2[[target] + features].dropna()

X = df_clean[features]
y = df_clean[target]

# 4. Feature Selection Strategy: Multicollinearity Check via Correlation Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(X.correlation() if hasattr(X, 'correlation') else X.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix of Predictive Building Environment Features")
plt.show()

# 5. Split Dataset into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Add constant vector for the intercept term (beta_0) as required by statsmodels
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

# 6. Fit the Ordinary Least Squares (OLS) Linear Regression Model
model = sm.OLS(y_train, X_train_sm).fit()

# Print comprehensive model summary (p-values, R-squared, t-statistics)
print(model.summary())

# 7. Model Evaluation and Performance Metrics
y_pred = model.predict(X_test_sm)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- Testing Performance Metrics ---")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Coefficient of Determination ($R^2$): {r2:.4f}")

# 8. Diagnostic Plots: Checking Linear Regression Assumptions
residuals = y_test - y_pred

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
# Residuals vs Fitted values (Homoscedasticity evaluation)
sns.scatterplot(x=y_pred, y=residuals, ax=ax[0])
ax[0].axhline(y=0, color='r', linestyle='--')
ax[0].set_title('Residuals vs Predicted Values')
ax[0].set_xlabel('Predicted Energy Demand')
ax[0].set_ylabel('Residuals')

# Normality of Residuals (Q-Q plot)
sm.qqplot(residuals, line='45', ax=ax[1])
ax[1].set_title('Normal Q-Q Plot')

plt.tight_layout()
plt.show()